# Axis Bank — Social Intelligence: data explorer

Reads `axis.db` directly (read-only). Nothing here writes to the database.

**Run everything:** Kernel → Restart & Run All, or Shift+Enter through the cells.

Jump to: [Overview](#overview) · [Sources](#sources) · [Sentiment](#sentiment) · [Risk set](#risk) · [New sources](#new) · [Top posts](#top) · [Threads](#threads) · [Your own SQL](#sql)

In [ ]:
import sqlite3
import pandas as pd

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 90)

DB = r"C:\Users\nandu\axis-sentiment-poc\axis.db"

# read-only URI so nothing in this notebook can modify the database
con = sqlite3.connect(f"file:{DB}?mode=ro", uri=True, timeout=10)

def q(sql, **kw):
    """Run SQL, get a DataFrame."""
    return pd.read_sql(sql, con, **kw)

print("connected:", DB)

<a id='overview'></a>
## 1. What's in the database

41 tables and views. `raw_posts` is every mention collected; `analysis` is the per-post AI read; `mart_*` are the pre-aggregated dashboard tables.

In [ ]:
names = q("""SELECT name, type FROM sqlite_master
              WHERE type IN ('table','view') AND name NOT LIKE 'sqlite_%'
              ORDER BY type, name""")

rows = []
for _, r in names.iterrows():
    try:
        n = q(f'SELECT count(*) c FROM "{r["name"]}"')["c"][0]
    except Exception:
        n = None
    rows.append({"object": r["name"], "type": r["type"], "rows": n})

inventory = pd.DataFrame(rows).sort_values("rows", ascending=False, na_position="last")
inventory.reset_index(drop=True)

<a id='sources'></a>
## 2. Where the data came from

Note the split: `analysis` has fewer rows than `raw_posts`. Anything fetched but not yet classified has no AI read, so it won't appear in the dashboard.

In [ ]:
src = q("""SELECT r.source,
                  count(*) AS collected,
                  sum(CASE WHEN a.source_id IS NOT NULL THEN 1 ELSE 0 END) AS classified,
                  round(avg(a.score), 3) AS avg_sentiment
           FROM raw_posts r
           LEFT JOIN analysis a ON r.source_id = a.source_id
           GROUP BY r.source ORDER BY collected DESC""")
src["unclassified"] = src["collected"] - src["classified"]
print(f"collected {src['collected'].sum():,} | classified {src['classified'].sum():,} "
      f"| awaiting classification {src['unclassified'].sum():,}")
src

In [ ]:
ax = src.set_index("source")[["classified", "unclassified"]].plot(
    kind="barh", stacked=True, figsize=(9, 5),
    color=["#5CB198", "#C4544F"])
ax.set_title("Mentions by source (green = classified, red = awaiting the AI pass)")
ax.set_xlabel("posts"); ax.set_ylabel("")
ax.figure.tight_layout()

<a id='sentiment'></a>
## 3. Sentiment, intent and urgency

`intent` is the most useful categorical axis — it separates a complaint from a question from a legal threat.

In [ ]:
cats = q("""SELECT intent,
                   count(*) AS posts,
                   round(avg(score), 3) AS avg_score,
                   sum(fraud_signal) AS fraud_flags,
                   sum(CASE WHEN urgency = 'critical' THEN 1 ELSE 0 END) AS critical
            FROM analysis GROUP BY intent ORDER BY posts DESC""")
cats

In [ ]:
fig = q("SELECT sentiment, count(*) n FROM analysis GROUP BY sentiment").set_index("sentiment")["n"].plot(
    kind="pie", autopct="%1.1f%%", figsize=(5, 5), ylabel="",
    colors=["#C4544F", "#717A94", "#5CB198"])
fig.set_title("Sentiment mix")

### How much of the corpus got a deep read?

The pipeline is a cost cascade: a lexicon model scores everything instantly and free, and only the negative/ambiguous half escalates to an LLM. Rows scored `vader-fast` have **no** aspect, emotion or routing detail — and every one of them reports `emotion='joy'`, which is a degenerate default, not a real reading.

In [ ]:
depth = q("""SELECT model,
                    count(*) AS posts,
                    round(avg(confidence), 3) AS avg_confidence,
                    count(DISTINCT emotion) AS distinct_emotions
             FROM analysis GROUP BY model ORDER BY posts DESC""")
depth

<a id='risk'></a>
## 4. The risk set — what actually needs action

Out of ~1,769 classified posts, only a small slice carries real liability. This is the set worth a human's time.

In [ ]:
risk = q("""SELECT r.source, r.author, r.created_at, r.url,
                   a.intent, a.urgency, a.score, a.fraud_type,
                   a.recommended_team, a.recommended_action,
                   substr(COALESCE(a.text_masked, r.text), 1, 130) AS excerpt
            FROM analysis a JOIN raw_posts r ON a.source_id = r.source_id
            WHERE a.intent IN ('fraud_report','legal_threat','churn_threat')
               OR a.fraud_signal = 1 OR a.churn_risk = 1
            ORDER BY CASE a.urgency WHEN 'critical' THEN 4 WHEN 'high' THEN 3
                                    WHEN 'medium' THEN 2 ELSE 1 END DESC,
                     a.score ASC""")
print(f"{len(risk)} posts carry fraud / churn / legal exposure")
risk.head(20)

In [ ]:
# Posts where the customer exposed their OWN personal data — always read the masked column.
pii = q("""SELECT r.source, r.author, a.pii_types,
                  substr(a.text_masked, 1, 120) AS masked_excerpt, r.url
           FROM analysis a JOIN raw_posts r ON a.source_id = r.source_id
           WHERE a.pii_present = 1""")
print(f"{len(pii)} posts contain customer PII (shown masked)")
pii.head(10)

<a id='new'></a>
## 5. The newest sources (added via Scrapling)

Trustpilot reviews carry a star rating in the text prefix and a real publish date. These are the freshest rows in the database.

In [ ]:
new = q("""SELECT source, author, substr(created_at,1,10) AS date,
                  substr(text,1,120) AS excerpt
           FROM raw_posts
           WHERE source IN ('trustpilot','gmaps','valuepickr',
                            'businessstandard','consumercomplaints')
           ORDER BY source, created_at DESC""")
print(new['source'].value_counts().to_dict())
new.head(25)

<a id='top'></a>
## 6. The loudest posts

Engagement metrics exist on Twitter only, so this ranks tweets — not the whole corpus.

In [ ]:
top = q("""SELECT r.author, r.retweet_count AS retweets, r.reply_count AS replies,
                  r.view_count AS views, a.sentiment, a.score, a.intent,
                  substr(r.text, 1, 110) AS excerpt, r.url
           FROM raw_posts r JOIN analysis a ON r.source_id = a.source_id
           WHERE r.view_count > 0
           ORDER BY r.view_count DESC LIMIT 15""")
top

<a id='threads'></a>
## 7. Conversation threads

Author-diversity separates a genuine pile-on (many different people) from one account reposting.

In [ ]:
threads = q("""SELECT r.conversation_id,
                      count(*) AS posts,
                      count(DISTINCT r.author) AS authors,
                      round(avg(a.score), 3) AS avg_score
               FROM raw_posts r JOIN analysis a ON r.source_id = a.source_id
               WHERE r.conversation_id IS NOT NULL AND r.conversation_id <> ''
               GROUP BY r.conversation_id HAVING posts > 1
               ORDER BY posts DESC""")
threads["kind"] = ["brigade / many voices" if a >= 0.7 * n else "single-actor repetition"
                   for n, a in zip(threads["posts"], threads["authors"])]
print(threads["kind"].value_counts().to_dict())
threads.head(15)

In [ ]:
# Read one full thread end to end — paste any conversation_id from the table above.
CONV = threads["conversation_id"].iloc[0]

q(f"""SELECT r.created_at, r.author, a.score, a.sentiment,
             substr(r.text, 1, 160) AS text
      FROM raw_posts r JOIN analysis a ON r.source_id = a.source_id
      WHERE r.conversation_id = '{CONV}'
      ORDER BY r.created_at""")

<a id='sql'></a>
## 8. Your own SQL

Edit and re-run. Some starting points:

```sql
SELECT * FROM mart_kpis;
SELECT * FROM mart_competitor_sov;
SELECT * FROM mart_team_queue ORDER BY open_items DESC;
SELECT title, size, avg_score FROM clusters WHERE avg_score < 0 ORDER BY size DESC LIMIT 20;
SELECT * FROM raw_posts WHERE text LIKE '%UPI%' LIMIT 20;
```

In [ ]:
q("""SELECT * FROM mart_kpis""")

In [ ]:
# Full-text search across every collected post
TERM = "UPI"

q(f"""SELECT r.source, r.author, a.sentiment, a.score, a.intent,
             substr(r.text, 1, 140) AS excerpt
      FROM raw_posts r LEFT JOIN analysis a ON r.source_id = a.source_id
      WHERE r.text LIKE '%{TERM}%'
      ORDER BY a.score LIMIT 25""")